In [1]:
import sklearn
print(sklearn.__version__)

import warnings
warnings.filterwarnings("ignore")
# Other plots (example)

from src.exp import (
    ExperimentConfig, ExperimentFacade,
    DataReadConfig, PlotManager
)

1.8.0


In [2]:
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    print("Intel sklearn patch enabled")
except ImportError:
    print("sklearnex not installed; using standard sklearn")


sklearnex not installed; using standard sklearn


In [3]:
data_cfg = DataReadConfig(
    root_dir="Dataset/data",
    recursive=True,
    exclude_filenames=["cclass.csv", "unclean focus.csv","unclean cclass.csv","focus.csv"],  # the excluded files
    add_source_column=False,        # enable this to make additional column filled with the original taken filenames
)

In [4]:
cfg = ExperimentConfig(
    outer_folds=5,
    inner_folds=5,
    n_trials=40,
    seed=42,
    log_target=True
)

In [5]:
models = ["LinearRegression", "DecisionTree", "RandomForest", "SVR", "XGBoost", "NeuralNetwork"]

In [6]:
exp = ExperimentFacade.from_folder(
    data_cfg=data_cfg,
    target="price",
    cfg=cfg,
    model_names=models,
    hparam_json="config/hyperparams.json"
)

# model, year, price, transmission, mileage, fuelType, tax, mpg, engineSize

[schema]
  numerical cols: ['year', 'mileage', 'tax', 'mpg', 'engineSize']
  categorical cols: ['model', 'transmission', 'fuelType']
  target col: ['price']
  mapping: {'model': 'model', 'year': 'year', 'price': 'price', 'transmission': 'transmission', 'mileage': 'mileage', 'fuelType': 'fuelType', 'tax': 'tax', 'mpg': 'mpg', 'engineSize': 'engineSize'}


In [ ]:
results = exp.run()

[I 2026-02-24 07:22:07,364] A new study created in memory with name: LinearRegression_OuterFold_1_residual_cfg_base
[I 2026-02-24 07:22:11,194] Trial 0 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:11,261] Trial 1 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:11,315] Trial 2 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:11,359] Trial 3 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:11,401] Trial 4 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:11,447] Trial 5 finished with value: 2119.6030740614124 and parameters: {}. Best is trial 0 with value: 2119.6030740614124.
[I 2026-02-24 07:22:

In [ ]:
display(exp.summary())
model_summary = exp.summary()["model"]

,model,R2_mean,R2_std,MAE_mean,MAE_std,MedAE_mean,MedAE_std,MSE_mean,MSE_std,RMSE_mean,RMSE_std
0,DecisionTree,0.947015,0.004777,1284.564100,20.977425,801.957289,13.403884,5.155763e+06,4.504636e+05,2268.843307,100.705935
1,DecisionTree+ElasticNet,0.947033,0.004778,1284.716853,20.911955,801.773077,13.791613,5.153990e+06,4.506138e+05,2268.451106,100.744807
2,DecisionTree+Huber,0.947028,0.004767,1284.191780,20.963612,801.494555,14.879842,5.154434e+06,4.493552e+05,2268.559168,100.454733
3,DecisionTree+PseudoHuber,0.947563,0.004401,1280.186754,15.008215,799.439825,12.191988,5.102984e+06,4.223503e+05,2257.379631,95.005086
4,LinearRegression,-2.606856,7.771180,2110.810040,122.029471,1332.976244,7.017524,3.529705e+08,7.606829e+08,11138.970659,16915.000996
5,NeuralNetwork,0.855378,0.100311,1830.907971,72.164376,1152.081836,61.983012,1.410562e+07,9.853634e+06,3616.302983,1133.564112
6,RandomForest,0.959471,0.003362,1133.608924,28.101011,726.758290,16.178567,3.944784e+06,3.311553e+05,1984.748215,83.354356
7,RandomForest+ElasticNet,0.959539,0.003399,1133.912797,28.214816,726.741367,15.160861,3.938128e+06,3.347015e+05,1983.036394,84.367644
8,RandomForest+Huber,0.957119,0.004926,1154.969756,13.543526,735.347621,8.280192,4.173526e+06,4.798110e+05,2040.187671,118.110686
9,RandomForest+PseudoHuber,0.961363,0.004407,1128.428626,16.448138,726.284264,9.470179,3.760864e+06,4.348629e+05,1936.666040,112.854954


In [ ]:
model_summary.to_list()

['DecisionTree',
 'DecisionTree+ElasticNet',
 'DecisionTree+Huber',
 'DecisionTree+PseudoHuber',
 'LinearRegression',
 'NeuralNetwork',
 'RandomForest',
 'RandomForest+ElasticNet',
 'RandomForest+Huber',
 'RandomForest+PseudoHuber',
 'SVR',
 'XGBoost+Huber',
 'XGBoost+PseudoHuber']

In [ ]:
sig = exp.significance(
    metric="R2",
    baseline="XGBoost+PseudoHuber",
    models=model_summary.to_list()
)

In [ ]:
display(sig)

,metric,baseline,model,paired_t_p,wilcoxon_p,n_outer_folds
2,R2,XGBoost+PseudoHuber,DecisionTree+Huber,0.000017,0.0625,5
1,R2,XGBoost+PseudoHuber,DecisionTree+ElasticNet,0.000017,0.0625,5
0,R2,XGBoost+PseudoHuber,DecisionTree,0.000017,0.0625,5
3,R2,XGBoost+PseudoHuber,DecisionTree+PseudoHuber,0.000026,0.0625,5
10,R2,XGBoost+PseudoHuber,SVR,0.000236,0.0625,5
8,R2,XGBoost+PseudoHuber,RandomForest+Huber,0.001312,0.0625,5
6,R2,XGBoost+PseudoHuber,RandomForest,0.003513,0.0625,5
7,R2,XGBoost+PseudoHuber,RandomForest+ElasticNet,0.003536,0.0625,5
5,R2,XGBoost+PseudoHuber,NeuralNetwork,0.071710,0.0625,5
9,R2,XGBoost+PseudoHuber,RandomForest+PseudoHuber,0.079200,0.0625,5


In [ ]:
model_new = [
    'RandomForest',
    'SVR',
    'XGBoost',
    'XGBoost+Huber',
    'XGBoost+PseudoHuber',
    'NeuralNetwork']

In [ ]:
shap_analyzer = exp.shap(models=model_new) #exp.shap(models=model_summary.to_list())

In [ ]:
shap_analyzer.available_models()

['RandomForest',
 'RandomForest+ElasticNet',
 'RandomForest+Huber',
 'RandomForest+PseudoHuber',
 'SVR',
 'XGBoost',
 'XGBoost+Huber',
 'XGBoost+PseudoHuber',
 'NeuralNetwork']

In [ ]:
for m in shap_analyzer.available_models():
    shap_analyzer.beeswarm(m)

Computing SHAP for RandomForest:   0%|          | 0/20 [00:00<?, ?it/s]

: 

: 

In [ ]:
exp.summary()

,model,R2_mean,R2_std,MAE_mean,MAE_std,MedAE_mean,MedAE_std,MSE_mean,MSE_std,RMSE_mean,RMSE_std
0,DecisionTree,0.947015,0.004777,1284.564100,20.977425,801.957289,13.403884,5.155763e+06,4.504636e+05,2268.843307,100.705935
1,DecisionTree+ElasticNet,0.947033,0.004778,1284.716853,20.911955,801.773077,13.791613,5.153990e+06,4.506138e+05,2268.451106,100.744807
2,DecisionTree+Huber,0.947028,0.004767,1284.191780,20.963612,801.494555,14.879842,5.154434e+06,4.493552e+05,2268.559168,100.454733
3,DecisionTree+PseudoHuber,0.947563,0.004401,1280.186754,15.008215,799.439825,12.191988,5.102984e+06,4.223503e+05,2257.379631,95.005086
4,LinearRegression,-2.606856,7.771180,2110.810040,122.029471,1332.976244,7.017524,3.529705e+08,7.606829e+08,11138.970659,16915.000996
5,NeuralNetwork,0.855378,0.100311,1830.907971,72.164376,1152.081836,61.983012,1.410562e+07,9.853634e+06,3616.302983,1133.564112
6,RandomForest,0.959471,0.003362,1133.608924,28.101011,726.758290,16.178567,3.944784e+06,3.311553e+05,1984.748215,83.354356
7,RandomForest+ElasticNet,0.959539,0.003399,1133.912797,28.214816,726.741367,15.160861,3.938128e+06,3.347015e+05,1983.036394,84.367644
8,RandomForest+Huber,0.957119,0.004926,1154.969756,13.543526,735.347621,8.280192,4.173526e+06,4.798110e+05,2040.187671,118.110686
9,RandomForest+PseudoHuber,0.961363,0.004407,1128.428626,16.448138,726.284264,9.470179,3.760864e+06,4.348629e+05,1936.666040,112.854954


In [ ]:
results.to_csv("outputs/experiment_results.csv", index=False)

In [ ]:
pm = PlotManager("outputs/figures/metrics")

In [ ]:
for metric in ["R2", "MAE", "MedAE", "MSE", "RMSE"]:
    is_ascending = metric != "R2" and metric != "NegMSE"
    #fig = plot_point_range_1(results, metric=metric, ascending=is_ascending)
    #fig.savefig(f"outputs/figures/metrics/point_range_{metric.lower()}", dpi=300, bbox_inches="tight")
    fig = pm.plot_point_range(results_df=results, metric=metric, ascending=is_ascending)
    pm.save_fig(fig, f"point_range_{metric.lower()}")

[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/figures/metrics/point_range_r2.png
[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/figures/metrics/point_range_mae.png
[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/figures/metrics/point_range_medae.png
[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/figures/metrics/point_range_mse.png
[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/figures/metrics/point_range_rmse.png


In [ ]:
for metric in ["R2", "MAE", "MedAE", "MSE", "RMSE"]:
    display(exp.significance_matrix(metric=metric))

,metric,model_a,model_b,paired_t_p,wilcoxon_p,n_outer_folds
31,R2,DecisionTree+Huber,XGBoost+Huber,0.000007,0.0625,5
21,R2,DecisionTree+ElasticNet,XGBoost+Huber,0.000007,0.0625,5
10,R2,DecisionTree,XGBoost+Huber,0.000007,0.0625,5
40,R2,DecisionTree+PseudoHuber,XGBoost+Huber,0.000008,0.0625,5
32,R2,DecisionTree+Huber,XGBoost+PseudoHuber,0.000017,0.0625,5
...,...,...,...,...,...,...
24,R2,DecisionTree+Huber,LinearRegression,0.364221,0.0625,5
3,R2,DecisionTree,LinearRegression,0.364222,0.0625,5
47,R2,LinearRegression,SVR,0.367423,0.0625,5
42,R2,LinearRegression,NeuralNetwork,0.369973,0.0625,5


,metric,model_a,model_b,paired_t_p,wilcoxon_p,n_outer_folds
41,MAE,DecisionTree+PseudoHuber,XGBoost+PseudoHuber,0.000003,0.0625,5
56,MAE,NeuralNetwork,XGBoost+PseudoHuber,0.000011,0.0625,5
22,MAE,DecisionTree+ElasticNet,XGBoost+PseudoHuber,0.000013,0.0625,5
11,MAE,DecisionTree,XGBoost+PseudoHuber,0.000013,0.0625,5
32,MAE,DecisionTree+Huber,XGBoost+PseudoHuber,0.000014,0.0625,5
...,...,...,...,...,...,...
13,MAE,DecisionTree+ElasticNet,DecisionTree+PseudoHuber,0.294332,0.1250,5
2,MAE,DecisionTree,DecisionTree+PseudoHuber,0.311386,0.1250,5
23,MAE,DecisionTree+Huber,DecisionTree+PseudoHuber,0.350433,0.3125,5
64,MAE,RandomForest+ElasticNet,RandomForest+PseudoHuber,0.773775,0.6250,5


,metric,model_a,model_b,paired_t_p,wilcoxon_p,n_outer_folds
45,MedAE,LinearRegression,RandomForest+Huber,5.959551e-10,0.0625,5
44,MedAE,LinearRegression,RandomForest+ElasticNet,3.508145e-08,0.0625,5
43,MedAE,LinearRegression,RandomForest,4.800095e-08,0.0625,5
46,MedAE,LinearRegression,RandomForest+PseudoHuber,5.225085e-08,0.0625,5
3,MedAE,DecisionTree,LinearRegression,8.272696e-08,0.0625,5
...,...,...,...,...,...,...
12,MedAE,DecisionTree+ElasticNet,DecisionTree+Huber,7.028137e-01,0.8125,5
0,MedAE,DecisionTree,DecisionTree+ElasticNet,7.463916e-01,0.6250,5
64,MedAE,RandomForest+ElasticNet,RandomForest+PseudoHuber,9.581951e-01,0.8125,5
59,MedAE,RandomForest,RandomForest+PseudoHuber,9.584959e-01,0.8125,5


,metric,model_a,model_b,paired_t_p,wilcoxon_p,n_outer_folds
31,MSE,DecisionTree+Huber,XGBoost+Huber,0.000002,0.0625,5
21,MSE,DecisionTree+ElasticNet,XGBoost+Huber,0.000002,0.0625,5
10,MSE,DecisionTree,XGBoost+Huber,0.000002,0.0625,5
40,MSE,DecisionTree+PseudoHuber,XGBoost+Huber,0.000004,0.0625,5
38,MSE,DecisionTree+PseudoHuber,RandomForest+PseudoHuber,0.000013,0.0625,5
...,...,...,...,...,...,...
24,MSE,DecisionTree+Huber,LinearRegression,0.364281,0.0625,5
3,MSE,DecisionTree,LinearRegression,0.364282,0.0625,5
47,MSE,LinearRegression,SVR,0.367478,0.0625,5
42,MSE,LinearRegression,NeuralNetwork,0.370004,0.0625,5


,metric,model_a,model_b,paired_t_p,wilcoxon_p,n_outer_folds
21,RMSE,DecisionTree+ElasticNet,XGBoost+Huber,0.000002,0.0625,5
10,RMSE,DecisionTree,XGBoost+Huber,0.000002,0.0625,5
31,RMSE,DecisionTree+Huber,XGBoost+Huber,0.000002,0.0625,5
32,RMSE,DecisionTree+Huber,XGBoost+PseudoHuber,0.000003,0.0625,5
22,RMSE,DecisionTree+ElasticNet,XGBoost+PseudoHuber,0.000004,0.0625,5
...,...,...,...,...,...,...
59,RMSE,RandomForest,RandomForest+PseudoHuber,0.318602,0.6250,5
47,RMSE,LinearRegression,SVR,0.330175,0.0625,5
64,RMSE,RandomForest+ElasticNet,RandomForest+PseudoHuber,0.330864,0.6250,5
42,RMSE,LinearRegression,NeuralNetwork,0.346823,0.0625,5


In [ ]:
exp.save_best_params()

[saved] /Users/macbook/Library/CloudStorage/GoogleDrive-nur.ichsan@gmail.com/My Drive/UNS/2023/Business Intelligence/Paper/Salomo/NewIJAI/outputs/best_params/best_params.json
